# Projeto - Marketing Funnel by Olist

## Objetivo
Construir a analise completa do funil de Marketing e Vendas, parte por parte, com foco em:

1. Marketing Funnel Performance Dashboard
2. Lead-to-Revenue Funnel

## Analises
- Leads por origem
- Conversão (MQL -> Deal Fechado)
- Tempo médio entre estágios (first_contact_date -> won_date)
- Taxa de abandono
- Receita por canal/segmento

In [58]:
import pandas as pd
import numpy as np
import plotly.express as px

## Etapa 1 - Carga dos Dados e Padronização
Carregando os dois datasets principais e padronizamos os campos de data.

In [59]:
df_closed = pd.read_csv('dataset/olist_closed_deals_dataset.csv')
df_mql = pd.read_csv('dataset/olist_marketing_qualified_leads_dataset.csv')

df_closed['won_date'] = pd.to_datetime(df_closed['won_date'], errors='coerce')
df_mql['first_contact_date'] = pd.to_datetime(df_mql['first_contact_date'], errors='coerce')

print('Closed deals:', df_closed.shape)
print('MQL:', df_mql.shape)

Closed deals: (842, 14)
MQL: (8000, 4)


In [60]:
display(df_closed.head(3))
display(df_mql.head(3))

print('Colunas closed:', list(df_closed.columns))
print('Colunas mql:', list(df_mql.columns))

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.00
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08 20:17:59,car_accessories,industry,eagle,NaN,NaN,NaN,reseller,NaN,0.00
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05 17:27:23,home_appliances,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.00


,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search


Colunas closed: ['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue']
Colunas mql: ['mql_id', 'first_contact_date', 'landing_page_id', 'origin']


In [61]:
quality = pd.DataFrame({
    'dataset': ['closed_deals', 'mql'],
    'linhas': [len(df_closed), len(df_mql)],
    'duplicados_mql_id': [df_closed['mql_id'].duplicated().sum(), df_mql['mql_id'].duplicated().sum()],
    'datas_nulas': [df_closed['won_date'].isna().sum(), df_mql['first_contact_date'].isna().sum()]
})

display(quality)

print('Periodo first_contact_date:', df_mql['first_contact_date'].min(), 'ate', df_mql['first_contact_date'].max())
print('Periodo won_date:', df_closed['won_date'].min(), 'ate', df_closed['won_date'].max())

,dataset,linhas,duplicados_mql_id,datas_nulas
0,closed_deals,842,0,0
1,mql,8000,0,0


Periodo first_contact_date: 2017-06-14 00:00:00 ate 2018-05-31 00:00:00
Periodo won_date: 2017-12-05 02:00:00 ate 2018-11-14 18:04:19


## Etapa 2 - Base Unificada do Funil
MQL com Closed Deals por mql_id para medir conversão, abandono e tempo de fechamento.

In [62]:
df_funnel = df_mql.merge(df_closed, how='left', on='mql_id', suffixes=('_mql', '_deal'))
df_funnel['is_won'] = df_funnel['won_date'].notna().astype(int)
df_funnel['days_to_close'] = (df_funnel['won_date'] - df_funnel['first_contact_date']).dt.days

display(df_funnel.head(5))
df_funnel[['mql_id', 'origin', 'first_contact_date', 'won_date', 'is_won', 'days_to_close', 'declared_monthly_revenue']].head(5)

,mql_id,first_contact_date,landing_page_id,origin,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,is_won,days_to_close
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.00,1,5.00


,mql_id,origin,first_contact_date,won_date,is_won,days_to_close,declared_monthly_revenue
0,dac32acd4db4c29c230538b72f8dd87d,social,2018-02-01,NaT,0,NaN,NaN
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,paid_search,2017-10-20,NaT,0,NaN,NaN
2,b4bc852d233dfefc5131f593b538befa,organic_search,2018-03-22,NaT,0,NaN,NaN
3,6be030b81c75970747525b843c1ef4f8,email,2018-01-22,NaT,0,NaN,NaN
4,5420aad7fec3549a85876ba1c529bd84,organic_search,2018-02-21,2018-02-26 19:58:54,1,5.00,0.00


In [63]:
total_leads = len(df_funnel)
total_wins = int(df_funnel['is_won'].sum())
total_lost = total_leads - total_wins
conv_rate = total_wins / total_leads if total_leads else np.nan
abandon_rate = total_lost / total_leads if total_leads else np.nan
avg_days = df_funnel.loc[df_funnel['is_won'] == 1, 'days_to_close'].mean()

kpi_geral = pd.DataFrame({
    'kpi': ['Leads Totais', 'Deals Fechados', 'Deals Perdidos/Abandonados', 'Taxa de Conversao', 'Taxa de Abandono', 'Tempo Medio p/ Fechar (dias)'],
    'valor': [total_leads, total_wins, total_lost, conv_rate, abandon_rate, avg_days]
})

display(kpi_geral)

,kpi,valor
0,Leads Totais,"8,000.00"
1,Deals Fechados,842.00
2,Deals Perdidos/Abandonados,"7,158.00"
3,Taxa de Conversao,0.11
4,Taxa de Abandono,0.89
5,Tempo Medio p/ Fechar (dias),48.44


In [64]:
df_funnel_is_won = df_funnel[df_funnel['is_won'].eq(1) & df_funnel['declared_monthly_revenue'].gt(0)]
df_funnel_is_won = df_funnel_is_won.sort_values('days_to_close', ascending=False)

print(len(df_funnel_is_won['declared_monthly_revenue']))
df_funnel_is_won[['mql_id', 'origin', 'first_contact_date', 'won_date', 'days_to_close', 'declared_monthly_revenue']].head(5)

45


,mql_id,origin,first_contact_date,won_date,days_to_close,declared_monthly_revenue
6641,33ce1e734d9d50629fa2c36769285d53,NaN,2017-07-11,2018-09-11 13:14:37,427.00,"130,000.00"
4706,c5b432382d5978b94676426a32725dff,paid_search,2017-09-25,2018-10-03 18:32:49,373.00,"100,000.00"
5759,b39ac02ff5021fed10cb9988a23d5d02,social,2017-08-17,2018-08-23 21:56:01,371.00,"300,000.00"
7484,a57bf18e19b6b17e3d4fbc20561e2055,organic_search,2017-09-01,2018-08-20 14:41:13,353.00,"50,000.00"
5943,d79c868179307f1cf78d0a12c56e2bf9,organic_search,2017-09-23,2018-08-30 12:32:15,341.00,"60,000.00"


In [65]:
df_funnel_is_won = df_funnel[df_funnel['is_won'].eq(1)]
df_funnel_is_won[['mql_id', 'origin', 'first_contact_date', 'won_date', 'days_to_close']].head(5)
print(df_funnel_is_won['days_to_close'].max())
print(df_funnel_is_won['days_to_close'].min())
print(df_funnel_is_won['days_to_close'].mean())
print(df_funnel_is_won['days_to_close'].median())
print(df_funnel_is_won['days_to_close'].std())
print(df_funnel_is_won['days_to_close'].quantile(0.9))

427.0
-2.0
48.44061757719715
14.0
75.32811722592339
161.89999999999998


In [66]:
df_funnel['ano_mes'] = df_funnel['first_contact_date'].dt.to_period('M').astype(str)
df_funnel = df_funnel[df_funnel['ano_mes'].ne("2017-06")]

mensal = (
    df_funnel.groupby('ano_mes', dropna=False)
    .agg(
        leads=('mql_id', 'count'),
        ganhos=('is_won', 'sum'),
        tempo_medio_dias=('days_to_close', 'mean')
    )
    .reset_index()
)
mensal['taxa_conversao'] = mensal['ganhos'] / mensal['leads']
mensal['taxa_abandono'] = 1 - mensal['taxa_conversao']
display(mensal.head(12))

,ano_mes,leads,ganhos,tempo_medio_dias,taxa_conversao,taxa_abandono
0,2017-07,239,2,398.00,0.01,0.99
1,2017-08,386,9,296.00,0.02,0.98
2,2017-09,312,7,283.71,0.02,0.98
3,2017-10,416,14,240.21,0.03,0.97
4,2017-11,445,18,155.11,0.04,0.96
5,2017-12,200,11,122.36,0.06,0.94
6,2018-01,1141,152,43.65,0.13,0.87
7,2018-02,1028,149,42.28,0.14,0.86
8,2018-03,1174,167,37.61,0.14,0.86
9,2018-04,1352,183,23.82,0.14,0.86


In [73]:
fig_vol = px.line(mensal, x='ano_mes', y=['leads', 'ganhos'], title='Evolução Mensal: Leads vs Deals Fechados', markers=True)
fig_vol.update_layout(xaxis_title='Ano-Mês', yaxis_title='Quantidade')
fig_vol.show()

fig_conv = px.line(mensal, x='ano_mes', y=['taxa_conversao', 'taxa_abandono'], title='Taxas Mensais: Conversão e Abandono', markers=True)
fig_conv.update_layout(xaxis_title='Ano-Mês', yaxis_title='Taxa')
fig_conv.show()

## Etapa 3 - Lead-to-Revenue por Origem
Volume, conversão e receita para identificar canais mais eficientes em resultado e não apenas em volume.

In [ ]:
origin_perf = (
    df_funnel.groupby('origin', dropna=False)
    .agg(
        leads=('mql_id', 'count'),
        ganhos=('is_won', 'sum'),
        receita_total=('declared_monthly_revenue', 'sum'),
        tempo_medio_dias=('days_to_close', 'mean')
    )
    .reset_index()
)
origin_perf['taxa_conversao'] = origin_perf['ganhos'] / origin_perf['leads']
origin_perf['taxa_abandono'] = 1 - origin_perf['taxa_conversao']
origin_perf['ticket_medio_deal'] = np.where(origin_perf['ganhos'] > 0, origin_perf['receita_total'] / origin_perf['ganhos'], np.nan)
origin_perf = origin_perf.sort_values(['receita_total', 'taxa_conversao'], ascending=False)

display(origin_perf)

,origin,leads,ganhos,receita_total,tempo_medio_dias,taxa_conversao,taxa_abandono,ticket_medio_deal
3,organic_search,2296,271,"51,426,000.00",50.00,0.12,0.88,"189,763.84"
6,paid_search,1586,195,"9,169,000.00",56.60,0.12,0.88,"47,020.51"
8,social,1350,75,"501,000.00",60.96,0.06,0.94,"6,680.00"
9,unknown,1098,179,"463,006.00",41.29,0.16,0.84,"2,586.63"
10,NaN,59,14,"150,000.00",49.21,0.24,0.76,"10,714.29"
0,direct_traffic,499,56,"60,000.00",31.12,0.11,0.89,"1,071.43"
2,email,492,15,"15,000.00",52.20,0.03,0.97,"1,000.00"
7,referral,284,24,0.00,32.54,0.08,0.92,0.00
1,display,117,6,0.00,10.33,0.05,0.95,0.00
5,other_publicities,65,3,0.00,39.33,0.05,0.95,0.00


In [71]:
fig_origin_leads = px.bar(origin_perf, x='origin', y='leads', title='Leads por Origem', text='leads')
fig_origin_leads.update_traces(textposition='outside')
fig_origin_leads.show()

fig_origin_conv = px.bar(origin_perf, x='origin', y='taxa_conversao', title='Taxa de Conversão por Origem', text='taxa_conversao')
fig_origin_conv.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig_origin_conv.show()

fig_origin_rev = px.bar(origin_perf, x='origin', y='receita_total', title='Receita Total por Origem', text='receita_total')
fig_origin_rev.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig_origin_rev.show()

## Etapa 4 - Analise de Conversão e Receita por Segmento e Tipo de Lead

In [ ]:
# Business_segment e lead_type vêm do dataset de closed deals.

non_won = df_funnel[df_funnel['is_won'].eq(0)]
pct_non_won_sem_segmento = non_won[['business_segment', 'lead_type']].isna().all(axis=1).mean()

seg_type = (
    df_funnel[df_funnel['is_won'].eq(1)]
    .groupby(['business_segment', 'lead_type'], dropna=False)
    .agg(
        deals_fechados=('mql_id', 'count'),
        receita_total=('declared_monthly_revenue', 'sum'),
        tempo_medio_dias=('days_to_close', 'mean')
    )
    .reset_index()
)

total_deals_fechados = seg_type['deals_fechados'].sum()
seg_type['participacao_deals'] = np.where(
    total_deals_fechados > 0,
    seg_type['deals_fechados'] / total_deals_fechados,
    np.nan
)
seg_type['ticket_medio_deal'] = np.where(
    seg_type['deals_fechados'] > 0,
    seg_type['receita_total'] / seg_type['deals_fechados'],
    np.nan
)

seg_type = seg_type.sort_values('receita_total', ascending=False)
display(seg_type.head(30))

,business_segment,lead_type,deals_fechados,receita_total,tempo_medio_dias,participacao_deals,ticket_medio_deal
45,construction_tools_house_garden,industry,13,"50,000,000.00",71.62,0.02,"3,846,153.85"
136,phone_mobile,online_big,3,"8,000,000.00",75.33,0.00,"2,666,666.67"
122,other,other,3,"626,000.00",253.67,0.00,"208,666.67"
48,construction_tools_house_garden,online_big,17,"450,000.00",59.76,0.02,"26,470.59"
126,pet,industry,6,"300,000.00",67.67,0.01,"50,000.00"
96,home_decor,online_small,11,"300,000.00",41.64,0.01,"27,272.73"
6,audio_video_electronics,online_medium,31,"270,000.00",65.39,0.04,"8,709.68"
94,home_decor,online_big,17,"220,000.00",48.71,0.02,"12,941.18"
85,health_beauty,online_small,12,"210,000.00",48.00,0.01,"17,500.00"
156,toys,online_big,2,"180,000.00",170.00,0.00,"90,000.00"


In [ ]:
pivot_revenue = pd.pivot_table(
    seg_type,
    index='business_segment',
    columns='lead_type',
    values='receita_total',
    aggfunc='sum',
    fill_value=0
)
display(pivot_revenue)

fig_seg = px.bar(seg_type.head(20), x='business_segment', y='receita_total', color='lead_type', title='Top Segmentos por Receita (quebrados por lead_type)')
fig_seg.show()

lead_type,industry,offline,online_beginner,online_big,online_medium,online_small,online_top,other
business_segment,,,,,,,,
air_conditioning,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
audio_video_electronics,0.00,0.00,0.00,0.00,"270,000.00",0.00,0.00,0.00
baby,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
bags_backpacks,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
bed_bath_table,0.00,0.00,0.00,0.00,"30,000.00",0.00,0.00,0.00
books,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
car_accessories,0.00,"110,000.00",0.00,0.00,"75,000.00",0.00,0.00,0.00
computers,0.00,0.00,0.00,"60,000.00","5,000.00","40,000.00",0.00,0.00
construction_tools_house_garden,"50,000,000.00","100,000.00",6.00,"450,000.00","15,000.00","130,000.00",0.00,0.00
